# PseudoCal — pipeline walkthrough

This notebook walks through the PseudoCal pipeline end to end, stage by stage:

1. **Camera frame** → 2. **monocular depth → pseudo-LiDAR** → 3. **bird's-eye-view pillars** (pseudo vs. real LiDAR) → 4. **the coarse-to-fine cascade**.

It runs **fully offline**: it uses the synthetic `DummyDepthEstimator` and randomly generated inputs so it executes without KITTI data or trained checkpoints. Every cell calls the *real* library APIs, so the spots where you would plug in a real KITTI frame and trained weights are marked clearly.

> **Requirements:** the project (`uv sync --extra dev`) plus `jupyter` and `matplotlib` for this notebook:
> `pip install jupyter matplotlib`.
>
> To run against real data and a trained model instead, see the **last section** and `cascade.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 1. A camera frame

We synthesise a small RGB image and a plausible pinhole intrinsic matrix. Replace `image_rgb` / `K` with a real KITTI `image_02` frame and its `calib_cam_to_cam` intrinsics to run on real data.

In [ ]:
H0, W0 = 192, 640
# A simple synthetic scene: gradient sky + textured ground + a couple of "objects".
image_rgb = np.zeros((H0, W0, 3), dtype=np.uint8)
image_rgb[: H0 // 2] = np.linspace(180, 120, H0 // 2)[:, None, None]  # sky
image_rgb[H0 // 2 :] = np.linspace(90, 40, H0 - H0 // 2)[:, None, None]  # road
image_rgb[H0 // 2 - 30 : H0 // 2, 120:200] = (200, 80, 80)  # an object
image_rgb[H0 // 2 - 50 : H0 // 2, 420:470] = (80, 200, 120)  # another object

K = np.array([[500.0, 0.0, W0 / 2], [0.0, 500.0, H0 / 2], [0.0, 0.0, 1.0]], np.float32)

plt.figure(figsize=(8, 3))
plt.imshow(image_rgb)
plt.title("input camera frame (synthetic)")
plt.axis("off");

## 2. Monocular depth → pseudo-LiDAR

PseudoCal lifts the image into 3-D by estimating a **metric** depth map and back-projecting it through the pinhole model. Here we use `DummyDepthEstimator` (offline); in a real run this is `depth=depth_anything` (Depth-Anything-V2) or `depth=glpn`.

A Canny **keep-mask** drops depth-discontinuity pixels before back-projection (otherwise "flying pixels" streak across object boundaries).

In [ ]:
from pseudocal.data.dataset import prepare_camera_input
from pseudocal.depth import DummyDepthEstimator
from pseudocal.pseudolidar import backproject

WIDTH = HEIGHT = 256
# Resize + normalise the image, scale K to match, and compute the Canny keep-mask.
image, edge_mask, K_scaled = prepare_camera_input(image_rgb, K, WIDTH, HEIGHT)

depth_model = DummyDepthEstimator().to(DEVICE)  # swap for the real estimator here
depth = depth_model.estimate(image[None].to(DEVICE))  # (1, 1, H, W) metric depth

pseudo = backproject(
    depth,
    torch.from_numpy(K_scaled)[None].to(DEVICE),
    keep_mask=edge_mask[None].to(DEVICE),
    stride=2,
    min_depth=1.0,
    max_depth=80.0,
    max_points=15000,
)  # (1, P, 4): [x, y, z, valid]
pseudo = pseudo[0].cpu().numpy()
pseudo = pseudo[pseudo[:, 3] > 0]  # drop padding / invalid rows
print("pseudo-LiDAR points:", len(pseudo))

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].imshow(depth[0, 0].cpu(), cmap="plasma")
ax[0].set_title("monocular metric depth")
ax[0].axis("off")
ax[1].imshow(edge_mask[0], cmap="gray")
ax[1].set_title("Canny keep-mask (white = kept)")
ax[1].axis("off");

## 3. Bird's-eye-view pillars

Both clouds — the **pseudo-LiDAR** (from the camera) and the **real LiDAR** (transformed by the current extrinsic estimate) — are encoded as BEV pillar images in the camera frame. PseudoPillars compares them in BEV, which is what makes it robust to large yaw errors. Below we visualise the BEV occupancy of each cloud (a simplified view of what the learned pillar encoder consumes).

In [ ]:
# A synthetic "real" LiDAR scan: points scattered in front of the sensor.
rng = np.random.default_rng(0)
n = 8000
real_pcl = np.zeros((n, 4), np.float32)
real_pcl[:, 0] = rng.uniform(-20, 20, n)  # x (right)
real_pcl[:, 2] = rng.uniform(2, 50, n)  # z (forward)
real_pcl[:, 1] = rng.uniform(-2, 2, n)  # y (up/down)
real_pcl[:, 3] = 1.0


def bev_occupancy(xyz, x_range=(-50, 50), z_range=(-50, 50), size=200):
    """Rasterise points onto an x-z grid (camera-frame bird's-eye view)."""
    hist, _, _ = np.histogram2d(xyz[:, 0], xyz[:, 2], bins=size, range=[x_range, z_range])
    return np.log1p(hist).T  # log scale, transpose so forward (z) is up


fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(bev_occupancy(pseudo[:, :3]), origin="lower", cmap="viridis")
ax[0].set_title("pseudo-LiDAR BEV (from camera)")
ax[1].imshow(bev_occupancy(real_pcl[:, :3]), origin="lower", cmap="magma")
ax[1].set_title("real-LiDAR BEV")
for a in ax:
    a.set_xlabel("x (right)")
    a.set_ylabel("z (forward)")

## 4. The cascade

At inference the three stages run in sequence, each predicting the *residual* decalibration and refining the running estimate:

```
T0 = T_init
T1 = D1^-1 . T0   (PseudoPillars — coarse, initialisation-free)
T2 = D2^-1 . T1   (UniCal-M     — medium refinement)
T3 = D3^-1 . T2   (UniCal-S     — fine refinement)
```

Below we build the **coarse stage** with a tiny, randomly-initialised model and run it through the cascade API on one `RawSample`. With random weights the output is *not* accurate — the point is to show the stage/cascade interface. The next section shows how to load trained checkpoints for all three stages.

In [ ]:
from unical.losses.combined import CombinedLoss
from unical.losses.regression import RegressionLoss
from unical.losses.spatial import SpatialLoss
from unical.models.backbone import MobileViTBackbone
from unical.models.head import SplitRegressionHead
from unical.utils.transform import Transform

from pseudocal.cascade.runner import CascadeCalibrator, PseudoPillarsStage, RawSample
from pseudocal.models.pseudopillars import PseudoPillars
from pseudocal.pillars import PillarGrid

# A small PseudoPillars model (random weights, Dummy depth — offline).
pillar_channels = 8
coarse = PseudoPillars(
    depth=DummyDepthEstimator(),
    backbone=MobileViTBackbone(
        img_channels=pillar_channels,
        lidar_channels=pillar_channels,
        image_size=64,
        pretrained=None,
        spatial_head=True,
    ),
    head=SplitRegressionHead(in_features=640, common_hidden=[], trans_hidden=[64], rot_hidden=[64]),
    loss=CombinedLoss(RegressionLoss(), SpatialLoss()),
    grid=PillarGrid(x_min=-8.0, x_max=8.0, z_min=0.0, z_max=16.0, pillar_size=0.25),
    pillar_channels=pillar_channels,
    backproject_stride=4,
    max_pseudo_points=2000,
).eval()

# A ground-truth extrinsic and a large decalibration to recover from.
T_gt = Transform.from_rotation_translation(
    np.eye(3, dtype=np.float32), np.array([0, 0, 0.1], np.float32)
)
T_decal = Transform.from_rotation_translation(
    np.eye(3, dtype=np.float32), np.array([0.4, 0, 0], np.float32)
)
raw = RawSample(
    image_rgb=image_rgb,
    pcl=real_pcl,
    K=K,
    T_gt=T_gt,
    T_init=T_decal @ T_gt,
)

calibrator = CascadeCalibrator([PseudoPillarsStage(coarse, width=64, height=64)], device=DEVICE)
T_pred = calibrator.calibrate(raw)
print("recovered extrinsic (random weights — illustrative only):")
print(np.round(T_pred.matrix, 3))

## Running on real KITTI data with trained checkpoints

The cells above are offline and use synthetic inputs. To evaluate the **full trained cascade** on real KITTI frames, train the three stages (see the README) and run the end-to-end script:

```bash
python cascade.py data_dir=/path/to/kitti_raw \
  +ckpt.pseudopillars=logs/checkpoints/pseudopillars/last.ckpt \
  +ckpt.unical_m=logs/checkpoints/unical_m/last.ckpt \
  +ckpt.unical_s=logs/checkpoints/unical_s/last.ckpt
```

Programmatically, the same wiring is: load each stage's checkpoint, build a `PseudoPillarsStage` + two `UniCalRefineStage`s, pass them to a `CascadeCalibrator`, and call `.calibrate(raw)` on a `RawSample` from `PseudoKittiDataset.raw_sample(i)` — exactly what `cascade.py` does.